# 215 — Embed Layer 2 Chunks via H2OGPTe

Computes a dense embedding for every `Chunk` node using H2OGPTe's `client.encode_for_retrieval()` (no Collection upload required), writes the vectors back as `Chunk.embedding`, and creates a Neo4j vector index so the agent can do cosine similarity search over MAS 626 paragraphs.

No OpenAI API is used.

**Prerequisites**
- `.env` with `H2OGPTE_URL`, `H2OGPTE_API_KEY`, and the usual `NEO4J_*` credentials
- Notebook 211 already run (so `Chunk` nodes exist in Neo4j)

**Output**
- `Chunk.embedding` property on every chunk (dense float list)
- Neo4j vector index `chunk_embeddings` (cosine similarity)
- Smoke-test query returning MAS 626 chunks most relevant to a natural-language prompt

## Imports and connections

In [1]:
import os, sys, time
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))
load_dotenv(REPO / '.env')

from h2ogpte import H2OGPTE
from src.graph.connection import Neo4jConnection

H2OGPTE_URL = os.getenv('H2OGPTE_URL')
H2OGPTE_API_KEY = os.getenv('H2OGPTE_API_KEY')
if not H2OGPTE_URL or not H2OGPTE_API_KEY:
    raise RuntimeError('Set H2OGPTE_URL and H2OGPTE_API_KEY in .env')

client = H2OGPTE(address=H2OGPTE_URL, api_key=H2OGPTE_API_KEY)
print('H2OGPTe client OK')

conn = Neo4jConnection().connect()
print('Neo4j connected')

Please install the correct version of H2OGPTE with `pip install h2ogpte==1.6.59`.
You can enable strict version checking by passing strict_version_check=True.
H2OGPTe client OK
Neo4j connected


## Pick an embedding model

Leave `EMBEDDING_MODEL = None` to use the server default, or set a specific model from `client.list_embedding_models()`. We detect the vector dimension from the first batch so the Neo4j index is created with the correct size regardless of which model is used.

In [2]:
available = client.list_embedding_models()
print('Available embedding models:')
for m in available:
    print(f'  - {m}')

EMBEDDING_MODEL = None   # None → server default; or pick one from the list above
BATCH_SIZE      = 32     # chunks per encode_for_retrieval call

Available embedding models:
  - BAAI/bge-large-en-v1.5
  - BAAI/bge-m3
  - h2oai/embeddinggemma-300m-qat-q8_0-unquantized
  - mixedbread-ai/mxbai-embed-large-v1
  - off


## Pull chunks from Neo4j

In [3]:
rows = conn.run_query('MATCH (c:Chunk) RETURN c.chunk_id AS chunk_id, c.text AS text ORDER BY c.chunk_id')
chunks = [(r['chunk_id'], r['text']) for r in rows if r['text']]
print(f'{len(chunks)} chunks to embed')

267 chunks to embed


## Embed in batches

`encode_for_retrieval` returns `list[list[float]]`. We check the first batch's dimension before creating the Neo4j vector index.

In [4]:
vectors = {}
dim = None
start = time.time()
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    ids   = [cid for cid, _ in batch]
    texts = [txt for _, txt in batch]
    kwargs = {'chunks': texts}
    if EMBEDDING_MODEL:
        kwargs['embedding_model'] = EMBEDDING_MODEL
    embs = client.encode_for_retrieval(**kwargs)
    if dim is None:
        dim = len(embs[0])
        print(f'Detected embedding dimension: {dim}')
    for cid, vec in zip(ids, embs):
        vectors[cid] = vec
    print(f'  embedded {len(vectors):>3} / {len(chunks)} ({time.time() - start:.1f}s)')

print(f'\nDone. {len(vectors)} vectors, dim={dim}.')

Detected embedding dimension: 1024
  embedded  32 / 267 (2.0s)
  embedded  64 / 267 (2.7s)
  embedded  96 / 267 (3.3s)
  embedded 128 / 267 (4.0s)
  embedded 160 / 267 (4.6s)
  embedded 192 / 267 (5.3s)
  embedded 224 / 267 (5.9s)
  embedded 256 / 267 (6.6s)
  embedded 267 / 267 (7.1s)

Done. 267 vectors, dim=1024.


## Write embeddings back to Neo4j

In [5]:
records = [{'chunk_id': cid, 'embedding': vec} for cid, vec in vectors.items()]
BATCH = 200
cypher = '''
UNWIND $records AS row
MATCH (c:Chunk {chunk_id: row.chunk_id})
CALL db.create.setNodeVectorProperty(c, 'embedding', row.embedding)
RETURN count(*) AS n
'''
total = 0
for i in range(0, len(records), BATCH):
    n = conn.run_query(cypher, {'records': records[i:i + BATCH]})[0]['n']
    total += n
print(f'Wrote {total} embedding properties')

Wrote 267 embedding properties


## Create the Neo4j vector index

Dimension is taken from the first embedding we computed. Cosine similarity matches what H2OGPTe's retrieval assumes.

In [7]:
conn.run_query(f'''
    CREATE VECTOR INDEX chunk_embeddings IF NOT EXISTS
    FOR (c:Chunk) ON (c.embedding)
    OPTIONS {{indexConfig: {{
        `vector.dimensions`: {dim},
        `vector.similarity_function`: 'cosine'
    }}}}
''')
print(f'Vector index chunk_embeddings created for {dim}-dim cosine search')

# Wait for the index to come online (small graph — usually <5s)
for _ in range(20):
    rows = conn.run_query(
        "SHOW INDEXES YIELD name, state WHERE name = 'chunk_embeddings' RETURN state"
    )
    state = rows[0]['state'] if rows else 'UNKNOWN'
    if state == 'ONLINE':
        break
    time.sleep(1)
print(f'Index state: {state}')

Vector index chunk_embeddings created for 1024-dim cosine search
Index state: ONLINE


## Smoke test — semantic search over MAS 626

Encode a natural-language question, then use Neo4j's vector index to return the most semantically relevant MAS 626 paragraph chunks. This is the query shape the agent's `retrieve_typology_chunks` tool will use at runtime.

In [8]:
QUERY = 'customer due diligence requirements for shell companies and nominee directors'

kwargs = {'chunks': [QUERY]}
if EMBEDDING_MODEL:
    kwargs['embedding_model'] = EMBEDDING_MODEL
q_vec = client.encode_for_retrieval(**kwargs)[0]

rows = conn.run_query('''
    CALL db.index.vector.queryNodes('chunk_embeddings', 5, $q_vec)
    YIELD node, score
    RETURN node.chunk_id AS chunk_id,
           node.paragraph AS paragraph,
           substring(node.text, 0, 200) AS snippet,
           score
    ORDER BY score DESC
''', {'q_vec': q_vec})

print(f'Query: "{QUERY}"\n')
for r in rows:
    print(f'[{r["score"]:.3f}]  para {r["paragraph"]}  ({r["chunk_id"]})')
    print(f'         {r["snippet"]}...\n')

Query: "customer due diligence requirements for shell companies and nominee directors"

[0.862]  para 6.28  (6.28-c0)
         A bank shall implement the policies and procedures referred to in paragraph 6.27 when establishing business relations with a customer and when conducting ongoing due diligence....

[0.859]  para 6.25  (6.25-c0)
         Where there are any reasonable grounds for suspicion that existing business relations with a customer are connected with money laundering or terrorism financing, and where the bank considers it approp...

[0.856]  para 6.33  (6.33-c0)
         A bank may establish business relations with a customer before completing the verification of the identity of the customer as required by paragraph 6.9, natural persons appointed to act on behalf of t...

[0.856]  para 3.1  (3.1-c0)
         This Notice is based on the following principles, which shall serve as a guide for all banks in the conduct of their operations and business activities: (a) A bank sha

In [ ]:
conn.close()
print('Done.')